# Data Preprocessing

## Feature Scaling

Feature scaling is a method used to normalize the range of independent variables or features of data. In data processing, it is also known as data normalization and is generally performed during the data preprocessing step.

The goal of normalization is to change the values of numeric columns in the dataset to a common scale, without distorting differences in the ranges of values. For machine learning, every dataset does not require normalization. It is required only when features have different ranges.

Feature scaling applies to one column at a time.

The two most popular techniques for feature scaling are normalization and standardization.

### Normalization

Normalization works better when the features are normally distributed.

#### Rescaling (min-max normalization)

Also known as min-max scaling or min-max normalization, rescaling is the simplest method and consists in rescaling the range of features to scale the range in [0, 1] or [−1, 1]

$$x' = \frac{x - x_{min}}{x_{max} - x_{min}}$$

Rescaling can be applied to an arbitrary range [a, b] using the following formula, where $a$ and $b$ are the minimum and maximum values of the desired range:

$$x' = a + \frac{(x - x_{min})(b - a)}{x_{max} - x_{min}}$$

The values of the feature will be between 0 and 1.

#### Mean normalization

$$x' = \frac{x - \bar{x}}{x_{max} - x_{min}}$$

#### Standardization (Z-score normalization)

Data can include multiple dimensions. Feature standardization makes the values of each feature in the data have zero-mean (when subtracting the mean in the numerator) and unit-variance. This method is widely used for normalization in many machine learning algorithms (e.g., support vector machines, logistic regression, and artificial neural networks).

$$x' = \frac{x - \bar{x}}{\sigma}$$

All values inside the column will be between -3 and 3, approximately. Outliers can end up outside this range.

#### Robust scaling

Also known as *standardization using median and interquartile range (IQR)*, is designed to be robust to outliers. It scales features using the median and IQR as reference points instead of the mean and standard deviation.

Where $Q_1(x)$ and $Q_3(x)$ are the first and third quartiles of the data, respectively:

$$x' = \frac{x - Q_2(x)}{Q_3(x) - Q_1(x)}$$

#### Examples (data preprocessing)

In [2]:
import numpy as np
import pandas as pd

##### Example 1

1. Importing the dataset
2. Selecting the columns

In [3]:
dataset = pd.read_csv('data/preprocessing.csv')
# select all rows, all columns except the last one
x = dataset.iloc[:, :-1].values
# all rows, only the last column
y = dataset.iloc[:, -1].values

In [4]:
x, y

(array([['France', 44.0, 72000.0],
        ['Spain', 27.0, 48000.0],
        ['Germany', 30.0, 54000.0],
        ['Spain', 38.0, 61000.0],
        ['Germany', 40.0, nan],
        ['France', 35.0, 58000.0],
        ['Spain', nan, 52000.0],
        ['France', 48.0, 79000.0],
        ['Germany', 50.0, 83000.0],
        ['France', 37.0, 67000.0]], dtype=object),
 array(['No', 'Yes', 'No', 'No', 'Yes', 'Yes', 'No', 'Yes', 'No', 'Yes'],
       dtype=object))

3. Handling missing data

In [5]:
# handling missing numerical data
from sklearn.impute import SimpleImputer
# replace NaN values with the mean of the other values in the same column
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
# specify the columns with numerical values
imputer.fit(x[:, 1:3])
# replace the missing data with the parameter values specified in the previous steps
x[:, 1:3] = imputer.transform(x[:, 1:3])
x

array([['France', 44.0, 72000.0],
       ['Spain', 27.0, 48000.0],
       ['Germany', 30.0, 54000.0],
       ['Spain', 38.0, 61000.0],
       ['Germany', 40.0, 63777.77777777778],
       ['France', 35.0, 58000.0],
       ['Spain', 38.77777777777778, 52000.0],
       ['France', 48.0, 79000.0],
       ['Germany', 50.0, 83000.0],
       ['France', 37.0, 67000.0]], dtype=object)

4. Encoding categorical data: independent variable

   Encoding the independent variable by splitting the column with categorical data into multiple columns, one for each category.

    This process is called one-hot encoding: for each category, a binary vector is created.

    In this example, the first column is split into three columns: France, Spain, and Germany.

    France vector: [1, 0, 0], Spain vector: [0, 1, 0], Germany vector: [0, 0, 1]

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
# specify the encoding method, the column to encode, and what to do with the other columns
# in this case, the other columns are left untouched (remainder='passthrough')
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [0])], remainder='passthrough')
# transform the data in a numpy array, then apply the encoding
x = np.array(ct.fit_transform(x))
x

array([[1.0, 0.0, 0.0, 44.0, 72000.0],
       [0.0, 0.0, 1.0, 27.0, 48000.0],
       [0.0, 1.0, 0.0, 30.0, 54000.0],
       [0.0, 0.0, 1.0, 38.0, 61000.0],
       [0.0, 1.0, 0.0, 40.0, 63777.77777777778],
       [1.0, 0.0, 0.0, 35.0, 58000.0],
       [0.0, 0.0, 1.0, 38.77777777777778, 52000.0],
       [1.0, 0.0, 0.0, 48.0, 79000.0],
       [0.0, 1.0, 0.0, 50.0, 83000.0],
       [1.0, 0.0, 0.0, 37.0, 67000.0]], dtype=object)

5. Encoding categorical data: dependent variable (the target label)

In [7]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)
y

array([0, 1, 0, 0, 1, 1, 0, 1, 0, 1])

6. Splitting the dataset into the Training set and Test set

In [8]:
from sklearn.model_selection import train_test_split
# the test set will contain 20% of the data, random_state is set to 1 for learning purposes
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 1)

In [9]:
x_train

array([[0.0, 0.0, 1.0, 38.77777777777778, 52000.0],
       [0.0, 1.0, 0.0, 40.0, 63777.77777777778],
       [1.0, 0.0, 0.0, 44.0, 72000.0],
       [0.0, 0.0, 1.0, 38.0, 61000.0],
       [0.0, 0.0, 1.0, 27.0, 48000.0],
       [1.0, 0.0, 0.0, 48.0, 79000.0],
       [0.0, 1.0, 0.0, 50.0, 83000.0],
       [1.0, 0.0, 0.0, 35.0, 58000.0]], dtype=object)

In [10]:
x_test

array([[0.0, 1.0, 0.0, 30.0, 54000.0],
       [1.0, 0.0, 0.0, 37.0, 67000.0]], dtype=object)

In [11]:
y_train

array([0, 1, 0, 0, 1, 1, 0, 1])

In [12]:
y_test

array([0, 1])

7. Feature Scaling

Note: the dummy variables (created by one-hot encoding) should **not** be scaled.

In [20]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
x_train = sc.fit_transform(x_train)
x_test = sc.transform(x_test)

In [21]:
x_train

array([[-0.77459667, -0.57735027,  1.29099445, -0.19159184, -1.07812594],
       [-0.77459667,  1.73205081, -0.77459667, -0.01411729, -0.07013168],
       [ 1.29099445, -0.57735027, -0.77459667,  0.56670851,  0.63356243],
       [-0.77459667, -0.57735027,  1.29099445, -0.30453019, -0.30786617],
       [-0.77459667, -0.57735027,  1.29099445, -1.90180114, -1.42046362],
       [ 1.29099445, -0.57735027, -0.77459667,  1.14753431,  1.23265336],
       [-0.77459667,  1.73205081, -0.77459667,  1.43794721,  1.57499104],
       [ 1.29099445, -0.57735027, -0.77459667, -0.74014954, -0.56461943]])

In [16]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
# fit the scaler to the training set and transform it
x_train[:, 3:] = sc.fit_transform(x_train[:, 3:])
# transform the test set but do not fit
x_test[:, 3:] = sc.transform(x_test[:, 3:])

In [17]:
x_train

array([[-0.77459667, -0.57735027,  1.29099445, -0.19159184, -1.07812594],
       [-0.77459667,  1.73205081, -0.77459667, -0.01411729, -0.07013168],
       [ 1.29099445, -0.57735027, -0.77459667,  0.56670851,  0.63356243],
       [-0.77459667, -0.57735027,  1.29099445, -0.30453019, -0.30786617],
       [-0.77459667, -0.57735027,  1.29099445, -1.90180114, -1.42046362],
       [ 1.29099445, -0.57735027, -0.77459667,  1.14753431,  1.23265336],
       [-0.77459667,  1.73205081, -0.77459667,  1.43794721,  1.57499104],
       [ 1.29099445, -0.57735027, -0.77459667, -0.74014954, -0.56461943]])

In [18]:
x_test

array([[-0.77459667,  1.73205081, -0.77459667, -1.46618179, -0.9069571 ],
       [ 1.29099445, -0.57735027, -0.77459667, -0.44973664,  0.20564034]])